# requires-grad-leaf-assert — worked example 3: Show That Detaching a Parameter Breaks the Leaf+requires_grad Contract

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `requires-grad-leaf-assert`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Calling `.detach()` on an `nn.Parameter` creates a new tensor that is a leaf but has `requires_grad=False`. This is a common mistake when trying to log parameter values — the detached tensor looks like a leaf, but the optimizer cannot update it. A well-placed assert reveals this immediately instead of letting training silently stall.

## Worked solution

**Step 1 — create an `nn.Parameter`.** Parameters are leaves with `requires_grad=True` by default. They pass both checks.

**Step 2 — detach the parameter.** `p.detach()` returns a new tensor sharing the same storage but with `requires_grad=False`. The result `.is_leaf` is True, but `.requires_grad` is False.

**Step 3 — show the assert fires on the detached version.** The detached tensor passes the `is_leaf` check but fails `requires_grad`.

**Step 4 — show the original still passes.** Detach creates a *new* tensor; the original `nn.Parameter` is unaffected.

**Step 5 — connect to the real-world bug.** If you accidentally store `detached_params = [p.detach() for p in model.parameters()]` and pass that list to an optimizer, the assert would immediately catch the error.

In [ ]:
import torch as t
import torch.nn as nn

def assert_optim_ready(p: t.Tensor) -> bool:
    assert p.is_leaf, f'shape={tuple(p.shape)}: non-leaf — optimizer will silently skip'
    assert p.requires_grad, f'shape={tuple(p.shape)}: requires_grad=False — no update possible'
    return True

# --- exercise and print ---
t.manual_seed(0)
param = nn.Parameter(t.randn(4))
detached = param.detach()

print('--- Original nn.Parameter ---')
print('  is_leaf:       ', param.is_leaf)           # True
print('  requires_grad: ', param.requires_grad)      # True
try:
    assert_optim_ready(param)
    print('  assert: PASS')
except AssertionError as e:
    print('  assert: FAIL -', e)

print('\n--- Detached version ---')
print('  is_leaf:       ', detached.is_leaf)         # True
print('  requires_grad: ', detached.requires_grad)   # False
try:
    assert_optim_ready(detached)
    print('  assert: PASS')
except AssertionError as e:
    print('  assert: FAIL -', e)  # should print FAIL

print('\nSharing storage?', param.data_ptr() == detached.data_ptr())  # True